# Importar librerías y configurar el entorno

In [15]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

print("Open3D version:", o3d.__version__)


Open3D version: 0.20.0


# Cargar y visualizar la nube de puntos original

In [16]:
# 1. Cargar el archivo PLY
file_path = "FARO_Scan_Full_Sub1_down3_with_normals.ply"
pcd = o3d.io.read_point_cloud(file_path)

print(f"La nube de puntos tiene {len(pcd.points)} puntos.")
print("¿Tiene colores RGB?", pcd.has_colors())
print("¿Tiene normales?", pcd.has_normals())

# 2. Visualización rápida interactiva (Se abrirá una ventana de Open3D)
o3d.visualization.draw_geometries([pcd], window_name="Nube Original", width=1024, height=768)

La nube de puntos tiene 1647021 puntos.
¿Tiene colores RGB? True
¿Tiene normales? True


libEGL warning: failed to get driver name for fd -1

libEGL warning: MESA-LOADER: failed to retrieve device information

libEGL warning: failed to get driver name for fd -1

MESA: error: ZINK: failed to choose pdev
libEGL warning: egl: failed to create dri2 screen


# Preprocesamiento (Downsampling y Filtrado de Ruido)

In [17]:
# Estimar normales si no las tiene
if not pcd.has_normals():
    print("Estimando normales...")
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
    )

# Reducción espacial (Voxel Downsampling) - ajusta voxel_size según la escala (metros)
voxel_size = 0.02
pcd_down = pcd.voxel_down_sample(voxel_size=voxel_size)
print(f"Puntos después del downsampling: {len(pcd_down.points)}")

# Filtrado estadístico para eliminar ruido flotante
pcd_filtered, ind = pcd_down.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print(f"Puntos después de remover outliers: {len(pcd_filtered.points)}")

Puntos después del downsampling: 1647021
Puntos después de remover outliers: 1629345


# Segmentación Estructural (Pisos y Techos con RANSAC)

In [18]:
# Copia de trabajo para ir extrayendo elementos
remaining_pcd = pcd_filtered

# Lista para guardar los segmentos encontrados
segments = []

# Supongamos que queremos extraer los 2 planos principales (ej. suelo y techo/pared grande)
for i in range(2):
    plane_model, inliers = remaining_pcd.segment_plane(
        distance_threshold=0.03, 
        ransac_n=3, 
        num_iterations=1000
    )
    [a, b, c, d] = plane_model
    print(f"Plano {i+1} detectado: {a:.2f}x + {b:.2f}y + {c:.2f}z + {d:.2f} = 0")
    
    # Extraer los puntos que pertenecen al plano
    plane_cloud = remaining_pcd.select_by_index(inliers)
    
    # Asignar un color distintivo (por ejemplo, gris claro para estructuras)
    plane_cloud.paint_uniform_color([0.7, 0.7, 0.7])
    segments.append(plane_cloud)
    
    # Actualizar la nube restante (restando los puntos del plano)
    remaining_pcd = remaining_pcd.select_by_index(inliers, invert=True)

print(f"Puntos restantes tras remover planos estructurales: {len(remaining_pcd.points)}")

Plano 1 detectado: -0.00x + 0.01y + 1.00z + 1.68 = 0
Plano 2 detectado: -0.00x + 0.00y + 1.00z + 1.64 = 0
Puntos restantes tras remover planos estructurales: 1347224


# Clustering de Objetos Discretos (DBSCAN)

In [19]:
# Aplicar DBSCAN
# eps: distancia máxima entre vecinos (ej. 5 cm). min_points: mínimo de puntos para formar un cluster.
labels = np.array(remaining_pcd.cluster_dbscan(eps=0.05, min_points=15, print_progress=True))

max_label = labels.max()
print(f"Se identificaron {max_label + 1} clusters de objetos distintos.")

# Asignar colores aleatorios a cada objeto detectado
cmap = plt.get_cmap("tab20")
colors = cmap(labels / (max_label if max_label > 0 else 1))

# Los puntos clasificados como ruido (label == -1) se pintan de negro o gris oscuro
colors[labels < 0] = [0.1, 0.1, 0.1, 1.0]

remaining_pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])

Se identificaron 0 clusters de objetos distintos.             ] 2%
Precompute neighbors.[========================================] 100%


# Combinar Resultados y Exportar

In [20]:
# Unir la nube de los planos estructurales con los objetos clusterizados
final_pcd = segments[0]
for seg in segments[1:]:
    final_pcd += seg
final_pcd += remaining_pcd

# Exportar a un nuevo archivo PLY
output_file = "nube_segmentada_resultado.ply"
o3d.io.write_point_cloud(output_file, final_pcd)
print(f"¡Nube de puntos segmentada guardada exitosamente como '{output_file}'!")

# Visualizar el resultado final
o3d.visualization.draw_geometries([final_pcd], window_name="Resultado Segmentado", width=1024, height=768)

¡Nube de puntos segmentada guardada exitosamente como 'nube_segmentada_resultado.ply'!


libEGL warning: failed to get driver name for fd -1

libEGL warning: MESA-LOADER: failed to retrieve device information

libEGL warning: failed to get driver name for fd -1

MESA: error: ZINK: failed to choose pdev
libEGL warning: egl: failed to create dri2 screen
